# Workspace Inventory – Analysis Queries

Run these against the `workspace_inventory_snapshot` Delta table to understand your workspace health.

Each query answers a specific governance question.

## Setup

In [ ]:
%%sql
-- Use the latest snapshot only (in case you have multiple runs)
CREATE OR REPLACE TEMP VIEW inventory AS
SELECT *
FROM workspace_inventory_snapshot
WHERE snapshot_id = (
    SELECT snapshot_id 
    FROM workspace_inventory_snapshot 
    ORDER BY snapshot_time_utc DESC 
    LIMIT 1
)

---
## Q1. Workspace overview — what do we have?

The first thing to understand: how many items of each type exist, and what's the overall health.

In [ ]:
%%sql
-- Item count by type, with governance flags per type
SELECT 
    type,
    COUNT(*) AS total_items,
    SUM(CAST(is_stale AS INT)) AS stale_items,
    SUM(CAST(is_unused_artifact AS INT)) AS unused_items,
    SUM(CAST(has_missing_owner AS INT)) AS no_owner,
    SUM(CAST(is_orphaned_model AS INT) + CAST(is_orphaned_endpoint AS INT)) AS orphaned,
    ROUND(AVG(CAST(cleanup_candidate_score AS DOUBLE)), 1) AS avg_cleanup_score
FROM inventory
GROUP BY type
ORDER BY total_items DESC

---
## Q2. Who owns what? — ownership distribution

Identify the top creators and find items with no owner (system-generated or orphaned).

In [ ]:
%%sql
-- Top 15 creators by item count, with their stale %
SELECT 
    created_by,
    COUNT(*) AS items_owned,
    SUM(CAST(is_stale AS INT)) AS stale_items,
    ROUND(SUM(CAST(is_stale AS INT)) * 100.0 / COUNT(*), 1) AS stale_pct,
    ROUND(AVG(CAST(days_since_modified AS DOUBLE)), 0) AS avg_days_since_modified
FROM inventory
WHERE created_by IS NOT NULL AND created_by != '' AND created_by != 'None'
GROUP BY created_by
ORDER BY items_owned DESC
LIMIT 15

---
## Q3. What's actually being used? — usage vs. modification

The most important governance question: which items have genuine usage (from Activity Events), which are only modified, and which are completely dormant.

In [ ]:
%%sql
-- Usage categories: actively used, modified-only, completely dormant
SELECT 
    CASE 
        WHEN last_used_date IS NOT NULL AND last_used_date != '' AND last_used_date != 'None'
            THEN 'Actively used (has access events)'
        WHEN CAST(days_since_modified AS INT) <= 90 
            THEN 'Recently modified (no access data)'
        WHEN CAST(days_since_modified AS INT) <= 180 
            THEN 'Modified 90-180 days ago'
        ELSE 'Dormant (>180 days no activity)'
    END AS usage_category,
    COUNT(*) AS item_count,
    ROUND(COUNT(*) * 100.0 / (SELECT COUNT(*) FROM inventory), 1) AS pct_of_total
FROM inventory
GROUP BY 1
ORDER BY item_count DESC

---
## Q4. Most accessed items — what's actually valuable?

Items with the highest access counts are the ones you must protect during any cleanup.

In [ ]:
%%sql
-- Top 20 most accessed items in the last 30 days
SELECT 
    name,
    type,
    created_by,
    CAST(access_count_30d AS INT) AS access_count_30d,
    CAST(unique_users_30d AS INT) AS unique_users_30d,
    CAST(days_since_last_used AS INT) AS days_since_last_used,
    CAST(days_since_modified AS INT) AS days_since_modified
FROM inventory
WHERE access_count_30d IS NOT NULL AND access_count_30d != '' AND access_count_30d != 'None'
ORDER BY CAST(access_count_30d AS INT) DESC
LIMIT 20

---
## Q5. Stale items breakdown — what's sitting idle?

Breaks down stale items by type and age band to prioritize cleanup.

In [ ]:
%%sql
-- Stale items by type and age band
SELECT 
    type,
    CASE 
        WHEN CAST(days_since_modified AS INT) > 365 THEN '5. >1 year'
        WHEN CAST(days_since_modified AS INT) > 270 THEN '4. 270-365 days'
        WHEN CAST(days_since_modified AS INT) > 180 THEN '3. 180-270 days'
        WHEN CAST(days_since_modified AS INT) > 90  THEN '2. 90-180 days'
        ELSE '1. <90 days'
    END AS age_band,
    COUNT(*) AS item_count
FROM inventory
WHERE CAST(is_stale AS INT) = 1
GROUP BY type, 2
ORDER BY type, age_band

---
## Q6. Top cleanup candidates — what should we review first?

Items with the highest composite cleanup scores. These combine staleness, unused status, missing owner, orphan status, and age.

In [ ]:
%%sql
-- Top 30 cleanup candidates with all governance flags
SELECT 
    name,
    type,
    created_by,
    CAST(cleanup_candidate_score AS INT) AS score,
    CAST(is_stale AS INT) AS stale,
    CAST(is_unused_artifact AS INT) AS unused,
    CAST(has_missing_owner AS INT) AS no_owner,
    CAST(is_orphaned_model AS INT) AS orphan_model,
    CAST(is_orphaned_endpoint AS INT) AS orphan_ep,
    CAST(is_duplicate_name AS INT) AS duplicate,
    CAST(days_since_modified AS INT) AS days_mod,
    CAST(days_since_last_used AS INT) AS days_used
FROM inventory
WHERE CAST(cleanup_candidate_score AS INT) >= 30
ORDER BY CAST(cleanup_candidate_score AS INT) DESC
LIMIT 30

---
## Q7. Orphaned items — what has lost its parent or purpose?

Semantic models with no report, SQL endpoints with no lakehouse/warehouse.

In [ ]:
%%sql
-- All orphaned items with context
SELECT 
    name,
    type,
    created_by,
    created_date,
    CAST(days_since_modified AS INT) AS days_since_modified,
    CAST(is_unused_artifact AS INT) AS unused,
    CAST(cleanup_candidate_score AS INT) AS score
FROM inventory
WHERE CAST(is_orphaned_model AS INT) = 1 
   OR CAST(is_orphaned_endpoint AS INT) = 1
ORDER BY CAST(cleanup_candidate_score AS INT) DESC

---
## Q8. Unused Power BI artifacts — confirmed no usage

These items were flagged by `list_unused_artifacts()` as having zero usage in Power BI metrics. Combined with their last accessed date from the same API.

In [ ]:
%%sql
-- Unused artifacts with their last known access date
SELECT 
    name,
    type,
    created_by,
    created_date,
    last_used_date,
    CAST(days_since_last_used AS INT) AS days_since_used,
    CAST(days_since_modified AS INT) AS days_since_mod,
    CAST(cleanup_candidate_score AS INT) AS score
FROM inventory
WHERE CAST(is_unused_artifact AS INT) = 1
ORDER BY CAST(cleanup_candidate_score AS INT) DESC

---
## Q9. Duplicate items — same name and type appearing more than once

Could indicate copy-paste development, failed deployments, or test artifacts left behind.

In [ ]:
%%sql
-- Duplicate items side by side
SELECT 
    a.name,
    a.type,
    a.id,
    a.created_by,
    a.created_date,
    a.last_modified,
    CAST(a.days_since_modified AS INT) AS days_mod
FROM inventory a
WHERE CAST(a.is_duplicate_name AS INT) = 1
ORDER BY a.name, a.type, a.last_modified DESC

---
## Q10. Items with no owner — who created these?

System-generated items, imported items, or items whose creator was deleted from the tenant.

In [ ]:
%%sql
-- Items with missing owner
SELECT 
    name,
    type,
    created_date,
    last_modified,
    CAST(days_since_modified AS INT) AS days_mod,
    CAST(is_stale AS INT) AS stale,
    CAST(cleanup_candidate_score AS INT) AS score
FROM inventory
WHERE CAST(has_missing_owner AS INT) = 1
ORDER BY type, name

---
## Q11. Safe-to-keep list — items that are actively used and should NOT be deleted

The inverse of cleanup candidates. Items with recent usage, active modifications, and known owners.

In [ ]:
%%sql
-- Items confirmed as actively used (do NOT delete)
SELECT 
    name,
    type,
    created_by,
    CAST(access_count_30d AS INT) AS access_30d,
    CAST(unique_users_30d AS INT) AS users_30d,
    CAST(days_since_last_used AS INT) AS days_since_used,
    CAST(days_since_modified AS INT) AS days_since_mod
FROM inventory
WHERE last_used_date IS NOT NULL 
  AND last_used_date != '' 
  AND last_used_date != 'None'
  AND CAST(is_stale AS INT) = 0
ORDER BY CAST(access_count_30d AS INT) DESC

---
## Q12. Creation timeline — when were items created?

Helps identify bulk-creation events (migrations, workshops, hackathons) and the age profile of the workspace.

In [ ]:
%%sql
-- Items created per month (where created_date is available)
SELECT 
    DATE_FORMAT(TO_TIMESTAMP(created_date), 'yyyy-MM') AS created_month,
    COUNT(*) AS items_created,
    COLLECT_SET(type) AS types_created
FROM inventory
WHERE created_date IS NOT NULL AND created_date != '' AND created_date != 'None'
GROUP BY 1
ORDER BY 1

---
## Q13. Modified-by analysis — who is actively working in this workspace?

Shows who has been modifying items recently, which helps identify active contributors vs. one-time creators.

In [ ]:
%%sql
-- Active modifiers (modified_by from Scanner API)
SELECT 
    modified_by,
    COUNT(*) AS items_modified,
    MIN(CAST(days_since_modified AS INT)) AS most_recent_mod_days,
    ROUND(AVG(CAST(days_since_modified AS INT)), 0) AS avg_days_since_mod
FROM inventory
WHERE modified_by IS NOT NULL AND modified_by != '' AND modified_by != 'None'
GROUP BY modified_by
ORDER BY items_modified DESC

---
## Q14. Governance health scorecard — single-number summary

One query that gives you the overall workspace health at a glance.

In [ ]:
%%sql
-- Workspace governance scorecard
SELECT
    COUNT(*) AS total_items,
    SUM(CASE WHEN CAST(is_stale AS INT) = 0 THEN 1 ELSE 0 END) AS active_items,
    SUM(CAST(is_stale AS INT)) AS stale_items,
    ROUND(SUM(CAST(is_stale AS INT)) * 100.0 / COUNT(*), 1) AS stale_pct,
    SUM(CAST(is_unused_artifact AS INT)) AS unused_artifacts,
    SUM(CAST(has_missing_owner AS INT)) AS no_owner_items,
    SUM(CAST(is_duplicate_name AS INT)) AS duplicate_items,
    SUM(CAST(is_orphaned_model AS INT)) AS orphaned_models,
    SUM(CAST(is_orphaned_endpoint AS INT)) AS orphaned_endpoints,
    SUM(CASE WHEN CAST(cleanup_candidate_score AS INT) >= 50 THEN 1 ELSE 0 END) AS high_risk_items,
    SUM(CASE WHEN CAST(cleanup_candidate_score AS INT) >= 30 AND CAST(cleanup_candidate_score AS INT) < 50 THEN 1 ELSE 0 END) AS medium_risk_items,
    SUM(CASE WHEN last_used_date IS NOT NULL AND last_used_date != '' AND last_used_date != 'None' THEN 1 ELSE 0 END) AS items_with_usage_data,
    ROUND(AVG(CAST(cleanup_candidate_score AS DOUBLE)), 1) AS avg_cleanup_score
FROM inventory

---
## Q15. Lakehouse/Warehouse ecosystem — parent-child relationships

Shows each Lakehouse and Warehouse alongside its auto-generated SQL endpoint, and flags any orphaned endpoints.

In [ ]:
%%sql
-- Lakehouse/Warehouse with their SQL endpoints
SELECT 
    COALESCE(p.name, e.name) AS item_name,
    p.type AS parent_type,
    CASE WHEN p.id IS NOT NULL THEN 'Yes' ELSE 'MISSING' END AS parent_exists,
    CASE WHEN e.id IS NOT NULL THEN 'Yes' ELSE 'No endpoint' END AS endpoint_exists,
    p.created_by AS parent_owner,
    CAST(p.days_since_modified AS INT) AS parent_days_mod,
    CAST(e.days_since_modified AS INT) AS endpoint_days_mod
FROM (
    SELECT * FROM inventory WHERE type IN ('Lakehouse', 'Warehouse')
) p
FULL OUTER JOIN (
    SELECT * FROM inventory WHERE type = 'SQLEndpoint'
) e ON LOWER(TRIM(p.name)) = LOWER(TRIM(e.name))
ORDER BY item_name